In [24]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()  # Load environment variables from .env file

True

In [25]:
client = Anthropic()
model = "claude-sonnet-4-0"

In [29]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

## basic api request

In [27]:
# messages = []

# add_user_message(messages, "what is the last function that select word in the end of a multilayer perceptron in gpt 2?")

# chat_response = chat(messages)
# print(chat_response)

# add_assistant_message(messages, chat_response)
# print(messages)

# add_user_message(messages, "What is the population of Paris?")
# chat_response = chat(messages)

# add_assistant_message(messages, chat_response)
# print(messages)

## system prompt

In [23]:
# system_prompt = """
# You are a code generator.
# Just provide the code requested, do not provide any explanations.
# """

# messages = []
# add_user_message(messages, "give me a function that adds two numbers in python")
# chat_response = chat(messages, system=system_prompt)
# print(chat_response)

## Temperature

In [41]:
def chat(messages, system=None, temperature=0.5, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text

## Response streaming

In [31]:
messages = []
add_user_message(messages, "what is France's population?")

event = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    temperature=0.5,
    stream=True
)

for chunk in event:
    print(chunk)


RawMessageStartEvent(message=Message(id='msg_01NW3LRcqvTzdmh1b6vvHATP', content=[], model='claude-sonnet-4-20250514', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=13, output_tokens=6, server_tool_use=None, service_tier='standard'), stop_details=None), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text="France's population is approximately ", type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='68 million people as of 2024. This includes about 65', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' million people in

In [34]:
messages = []
add_user_message(messages, "what is the last function that select word in the end of a multilayer perceptron in gpt 2?")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages,
    temperature=0.5
) as stream:
    for chunk in stream.text_stream:
        print(chunk, end="")
        pass

stream.get_final_message()

In GPT-2, the last function that selects words at the end of the multilayer perceptron is the **softmax function**.

Here's how it works:

1. **Final Linear Layer**: The last hidden states from the transformer layers are passed through a linear projection layer (often called the "language modeling head" or "lm_head") that maps the hidden dimension to the vocabulary size.

2. **Logits**: This produces raw logits - one score for each token in the vocabulary (typically 50,257 tokens for GPT-2).

3. **Softmax Function**: The softmax function is applied to these logits to convert them into a probability distribution:

```
P(token_i) = exp(logit_i) / Σ(exp(logit_j)) for all j in vocabulary
```

4. **Word Selection**: From this probability distribution, words can be selected using various strategies:
   - **Greedy decoding**: Select the token with highest probability
   - **Sampling**: Sample from the probability distribution
   - **Top-k sampling**: Sample from the k most likely tokens
   - 

ParsedMessage(id='msg_01SmgDzS4SWm3nVm5ie1epGX', content=[ParsedTextBlock(citations=None, text='In GPT-2, the last function that selects words at the end of the multilayer perceptron is the **softmax function**.\n\nHere\'s how it works:\n\n1. **Final Linear Layer**: The last hidden states from the transformer layers are passed through a linear projection layer (often called the "language modeling head" or "lm_head") that maps the hidden dimension to the vocabulary size.\n\n2. **Logits**: This produces raw logits - one score for each token in the vocabulary (typically 50,257 tokens for GPT-2).\n\n3. **Softmax Function**: The softmax function is applied to these logits to convert them into a probability distribution:\n\n```\nP(token_i) = exp(logit_i) / Σ(exp(logit_j)) for all j in vocabulary\n```\n\n4. **Word Selection**: From this probability distribution, words can be selected using various strategies:\n   - **Greedy decoding**: Select the token with highest probability\n   - **Samplin

## Structured Data

In [42]:
messages = []
add_user_message(messages, "Generate a very short event bridge as json")
add_assistant_message(messages, "```json")

response = chat(messages, stop_sequences=["```"])

In [43]:
response_json = response.strip("```").strip()
print(response_json)

{
  "Source": "myapp.orders",
  "DetailType": "Order Placed",
  "Detail": {
    "orderId": "12345",
    "customerId": "cust-789",
    "amount": 99.99
  }
}


In [46]:
messages = []
add_user_message(messages, "Generate three different sample AWS CLI commands. Each should be short and simple.")
add_assistant_message(messages, " Here are the commands in a single block without any comments ```bash")

response = chat(messages, stop_sequences=["```"])
print(response)


aws s3 ls

aws ec2 describe-instances

aws iam list-users

